# Advanced preprocessing

Keeps the speeches from 2015 (Paris Agreement) onwards and measures two things per speech: how often it mentions the
energy transition, and how cautious or committed that talk is. Caution is measured in three ways, so the methods can
be compared.

In [1]:
import os
import re
import numpy as np
import pandas as pd
import nltk
from nltk.tokenize import sent_tokenize

nltk.download('punkt')
nltk.download('punkt_tab')

import spacy


def load_spacy_model(name="en_core_web_sm"):
    try:
        return spacy.load(name, disable=["parser", "ner"])
    except OSError:
        spacy.cli.download(name)
        return spacy.load(name, disable=["parser", "ner"])


nlp = load_spacy_model()

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\30690\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\30690\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


## Load speeches from 2015 onwards

In [2]:
df = pd.read_parquet('../datasets/speeches_simple_clean.parquet')
df = df[df.index.get_level_values('Year') >= 2015]

print(df.shape)
df.head()

(2128, 10)


Session  \
Year ISO-alpha3 Code            
2015 AFG                   70   
     AGO                   70   
     ALB                   70   
     AND                   70   
     ARE                   70   

                                                                 Speech  \
Year ISO-alpha3 Code                                                      
2015 AFG              It gives me great pleasure, on behalf of the I...   
     AGO              At the outset, on behalf of the President of A...   
     ALB              One year ago, Pope Francis began his visits ar...   
     AND              It is my honour to represent my country, the P...   
     ARE              It is my pleasure to congratulate Mr. Mogens L...   

                                              SpeakerName  \
Year ISO-alpha3 Code                                        
2015 AFG                            Mr. Abdullah Abdullah   
     AGO                      Mr. Manuel Domingos Vicente   
     ALB                                     Mr. Edi Rama   
     AND                           Mr. Antoni Martí Petit   
     ARE              Sheikh Abdullah Bin Zayed Al Nahyan   

                                              Post       Country or Area  \
Year ISO-alpha3 Code                                                       
2015 AFG                   Chief Executive Officer           Afghanistan   
     AGO                            vice-President                Angola   
     ALB                            Prime minister               Albania   
     AND                        Head of Government               Andorra   
     ARE              Minister for Foreign Affairs  United Arab Emirates   

                     Region Name     Sub-region Name  \
Year ISO-alpha3 Code                                   
2015 AFG                    Asia       Southern Asia   
     AGO                  Africa  Sub-Saharan Africa   
     ALB                  Europe     Southern Europe   
     AND                  Europe     Southern Europe   
     ARE                    Asia        Western Asia   

                     Least Developed Countries (LDC)  \
Year ISO-alpha3 Code                                   
2015 AFG                                           x   
     AGO                                           x   
     ALB                                        None   
     AND                                        None   
     ARE                                        None   

                                                            SpeechClean  \
Year ISO-alpha3 Code                                                      
2015 AFG              It gives me great pleasure, on behalf of the I...   
     AGO              At the outset, on behalf of the President of A...   
     ALB              One year ago, Pope Francis began his visits ar...   
     AND              It is my honour to represent my country, the P...   
     ARE              It is my pleasure to congratulate Mr. Mogens L...   

                      WordCount  
Year ISO-alpha3 Code             
2015 AFG                   1741  
     AGO                   1644  
     ALB                   1097  
     AND                   2522  
     ARE                   1673

## Keyword and phrase lists

Regular expressions for energy-transition terms, and phrase lists for hedging ("we are considering", "gradually") and
commitment ("we will", "we pledge").

In [3]:
TRANSITION_FRAGMENTS = [
    r"energy\s+transitions?",
    r"net[\s-]?zero",
    r"renewable\s+(?:energy|power|sources)",
    r"renewables",
    r"decarboni[sz]ation",
    r"decarboni[sz]e[ds]?",
    r"clean\s+energy",
    r"low[\s-]carbon",
    r"carbon\s+neutral(?:ity)?",
    r"climate\s+neutral(?:ity)?",
    r"phas(?:e|ing)[\s-]?out\s+(?:of\s+)?fossil\s+fuels?",
    r"fossil\s+fuel\s+phase[\s-]?out",
    r"green\s+energy",
    r"green\s+transition",
    r"just\s+transition",
    r"solar\s+(?:energy|power)",
    r"wind\s+(?:energy|power)",
    r"clean\s+power",
    r"sustainable\s+energy",
    r"zero[\s-]?emissions?",
    r"carbon[\s-]?free",
    r"transition\s+to\s+(?:clean|renewable|green)\s+energy",
]

HEDGE_PHRASES = [
    "we will consider", "we are considering", "we are exploring", "we are studying",
    "we intend to explore", "look into", "looking into", "may consider", "could consider",
    "plan to explore", "gradually", "in due course", "over time", "step by step",
    "where possible", "as appropriate", "we hope to", "we aim to",
    "subject to available resources", "in the coming years", "we are open to", "remains a challenge",
]

COMMIT_PHRASES = [
    "we will", "we are committed to", "we pledge", "we pledge to",
    "has set a target of", "we have set a target", "we commit to", "we will achieve",
    "we will implement", "we are implementing", "our target is", "our goal is to achieve",
    "we will reduce", "we will phase out", "by 2030 we will", "by 2050 we will",
    "we guarantee", "we are determined to", "we vow to", "concrete steps to achieve",
]


def phrases_to_pattern(phrases):
    escaped = [re.escape(p).replace(r"\ ", r"\s+") for p in phrases]
    escaped.sort(key=len, reverse=True)
    return re.compile(r"\b(?:" + "|".join(escaped) + r")\b", re.IGNORECASE)


TRANSITION_PATTERN = re.compile(
    "|".join(sorted(TRANSITION_FRAGMENTS, key=len, reverse=True)), re.IGNORECASE
)
HEDGE_PATTERN = phrases_to_pattern(HEDGE_PHRASES)
COMMIT_PATTERN = phrases_to_pattern(COMMIT_PHRASES)

HEDGE_MODALS = {"may", "might", "could"}
COMMIT_MODALS = {"will", "shall", "must"}

HEDGE_VERB_LEMMAS = {
    "consider", "explore", "hope", "intend", "aim", "study", "plan", "examine", "review", "contemplate",
}
COMMIT_VERB_LEMMAS = {
    "commit", "pledge", "achieve", "reduce", "implement", "deliver", "guarantee",
    "fulfil", "fulfill", "accelerate", "ensure",
}

## Count transition mentions

In [4]:
df['transition_mention_count'] = df['SpeechClean'].apply(
    lambda text: len(list(TRANSITION_PATTERN.finditer(text)))
)

print(df['transition_mention_count'].describe())

count    2128.000000
mean        0.671992
std         1.367060
min         0.000000
25%         0.000000
50%         0.000000
75%         1.000000
max        20.000000
Name: transition_mention_count, dtype: float64


## Caution method 1: sentence window

We split each speech into sentences, keep the ones that mention the transition, and count the hedge and commitment
phrases in them.

`mask_and_count` counts the phrases and then blanks them out. Hedges are counted first, so the same words are never
also counted as a commitment.

For each speech we also keep one example sentence, only to check the results by hand later.

In [5]:
def mask_and_count(text, pattern):
    matches = list(pattern.finditer(text))
    masked = text
    for m in matches:
        masked = masked[:m.start()] + (" " * (m.end() - m.start())) + masked[m.end():]
    return len(matches), masked

In [6]:
sentence_hedge_counts = []
sentence_commit_counts = []
qualifying_sentences = {}
example_sentence_candidates = {}

for row_pos, speech in enumerate(df['SpeechClean']):
    sentences = sent_tokenize(speech)
    row_qualifying = [s for s in sentences if TRANSITION_PATTERN.search(s)]
    qualifying_sentences[row_pos] = row_qualifying

    hedge_total = 0
    commit_total = 0
    best_example = None
    fallback_example = row_qualifying[0] if row_qualifying else None

    for sentence in row_qualifying:
        hedge_n, masked = mask_and_count(sentence, HEDGE_PATTERN)
        commit_n, _ = mask_and_count(masked, COMMIT_PATTERN)
        hedge_total += hedge_n
        commit_total += commit_n
        if best_example is None and (hedge_n > 0 or commit_n > 0):
            best_example = sentence

    sentence_hedge_counts.append(hedge_total)
    sentence_commit_counts.append(commit_total)
    example_sentence_candidates[row_pos] = best_example if best_example is not None else fallback_example

df['sentence_hedge_count'] = sentence_hedge_counts
df['sentence_commit_count'] = sentence_commit_counts

print(df[['sentence_hedge_count', 'sentence_commit_count']].describe())

       sentence_hedge_count  sentence_commit_count
count           2128.000000            2128.000000
mean               0.006579               0.031485
std                0.080862               0.213431
min                0.000000               0.000000
25%                0.000000               0.000000
50%                0.000000               0.000000
75%                0.000000               0.000000
max                1.000000               4.000000


## Caution method 2: 15-word window

The same count, but only in the 15 words before and after each mention. A sentence can be long and talk about other
things, so this window is stricter. `word_tokens_with_offsets` and `extract_window` cut out the window.

In [7]:
def word_tokens_with_offsets(text):
    return [(m.group(), m.start(), m.end()) for m in re.finditer(r"\S+", text)]


def extract_window(text, tokens, match_start, match_end, radius=15):
    first_idx = next(i for i, (_, s, e) in enumerate(tokens) if e > match_start)
    last_idx = next(i for i in range(len(tokens) - 1, -1, -1) if tokens[i][1] < match_end)
    lo = max(0, first_idx - radius)
    hi = min(len(tokens) - 1, last_idx + radius)
    window_start = tokens[lo][1]
    window_end = tokens[hi][2]
    return text[window_start:window_end]

In [8]:
word15_hedge_counts = []
word15_commit_counts = []

for speech in df['SpeechClean']:
    tokens = word_tokens_with_offsets(speech)
    hedge_total = 0
    commit_total = 0
    for m in TRANSITION_PATTERN.finditer(speech):
        window_text = extract_window(speech, tokens, m.start(), m.end())
        hedge_n, masked = mask_and_count(window_text, HEDGE_PATTERN)
        commit_n, _ = mask_and_count(masked, COMMIT_PATTERN)
        hedge_total += hedge_n
        commit_total += commit_n
    word15_hedge_counts.append(hedge_total)
    word15_commit_counts.append(commit_total)

df['word15_hedge_count'] = word15_hedge_counts
df['word15_commit_count'] = word15_commit_counts

print(df[['word15_hedge_count', 'word15_commit_count']].describe())

       word15_hedge_count  word15_commit_count
count         2128.000000          2128.000000
mean             0.011278             0.039474
std              0.122137             0.251637
min              0.000000             0.000000
25%              0.000000             0.000000
50%              0.000000             0.000000
75%              0.000000             0.000000
max              2.000000             4.000000


## Caution method 3: grammar (spaCy)

Methods 1 and 2 only find phrases from our lists. Here spaCy tags each word in the transition sentences, and we count
hedging modal verbs (may, might, could) against committing ones (will, shall, must), plus hedging verbs (e.g. consider,
explore, hope) and committing verbs (e.g. commit, pledge, reduce). This also catches sentences like "we might reduce
emissions".

In [9]:
flat_sentences = []
flat_row_positions = []
for row_pos, sentences in qualifying_sentences.items():
    for sentence in sentences:
        flat_sentences.append(sentence)
        flat_row_positions.append(row_pos)

print(len(flat_sentences), "qualifying sentences across", df.shape[0], "speeches")

1264 qualifying sentences across 2128 speeches


In [10]:
pos_hedge_counts = np.zeros(len(df), dtype=int)
pos_commit_counts = np.zeros(len(df), dtype=int)

for row_pos, doc in zip(flat_row_positions, nlp.pipe(flat_sentences, batch_size=200)):
    hedge_hit = 0
    commit_hit = 0
    for token in doc:
        if token.tag_ == "MD":
            if token.lower_ in HEDGE_MODALS:
                hedge_hit += 1
            elif token.lower_ in COMMIT_MODALS:
                commit_hit += 1
        elif token.pos_ == "VERB":
            lemma = token.lemma_.lower()
            if lemma in HEDGE_VERB_LEMMAS:
                hedge_hit += 1
            elif lemma in COMMIT_VERB_LEMMAS:
                commit_hit += 1
    pos_hedge_counts[row_pos] += hedge_hit
    pos_commit_counts[row_pos] += commit_hit

df['pos_hedge_count'] = pos_hedge_counts
df['pos_commit_count'] = pos_commit_counts

print(df[['pos_hedge_count', 'pos_commit_count']].describe())

       pos_hedge_count  pos_commit_count
count      2128.000000       2128.000000
mean          0.039944          0.336466
std           0.216400          0.963629
min           0.000000          0.000000
25%           0.000000          0.000000
50%           0.000000          0.000000
75%           0.000000          0.000000
max           3.000000         17.000000


## Ratios and net caution

The counts are divided by the number of mentions. Net caution is the hedge ratio minus the commitment ratio; positive
means cautious.

In [11]:
df['example_sentence'] = [example_sentence_candidates[i] for i in range(len(df))]

denom = df['transition_mention_count'].replace(0, np.nan)
for prefix in ['sentence', 'word15', 'pos']:
    df[f'{prefix}_hedge_ratio'] = df[f'{prefix}_hedge_count'] / denom
    df[f'{prefix}_commit_ratio'] = df[f'{prefix}_commit_count'] / denom
    df[f'{prefix}_net_caution'] = df[f'{prefix}_hedge_ratio'] - df[f'{prefix}_commit_ratio']

df.head()

Session  \
Year ISO-alpha3 Code            
2015 AFG                   70   
     AGO                   70   
     ALB                   70   
     AND                   70   
     ARE                   70   

                                                                 Speech  \
Year ISO-alpha3 Code                                                      
2015 AFG              It gives me great pleasure, on behalf of the I...   
     AGO              At the outset, on behalf of the President of A...   
     ALB              One year ago, Pope Francis began his visits ar...   
     AND              It is my honour to represent my country, the P...   
     ARE              It is my pleasure to congratulate Mr. Mogens L...   

                                              SpeakerName  \
Year ISO-alpha3 Code                                        
2015 AFG                            Mr. Abdullah Abdullah   
     AGO                      Mr. Manuel Domingos Vicente   
     ALB                                     Mr. Edi Rama   
     AND                           Mr. Antoni Martí Petit   
     ARE              Sheikh Abdullah Bin Zayed Al Nahyan   

                                              Post       Country or Area  \
Year ISO-alpha3 Code                                                       
2015 AFG                   Chief Executive Officer           Afghanistan   
     AGO                            vice-President                Angola   
     ALB                            Prime minister               Albania   
     AND                        Head of Government               Andorra   
     ARE              Minister for Foreign Affairs  United Arab Emirates   

                     Region Name     Sub-region Name  \
Year ISO-alpha3 Code                                   
2015 AFG                    Asia       Southern Asia   
     AGO                  Africa  Sub-Saharan Africa   
     ALB                  Europe     Southern Europe   
     AND                  Europe     Southern Europe   
     ARE                    Asia        Western Asia   

                     Least Developed Countries (LDC)  \
Year ISO-alpha3 Code                                   
2015 AFG                                           x   
     AGO                                           x   
     ALB                                        None   
     AND                                        None   
     ARE                                        None   

                                                            SpeechClean  \
Year ISO-alpha3 Code                                                      
2015 AFG              It gives me great pleasure, on behalf of the I...   
     AGO              At the outset, on behalf of the President of A...   
     ALB              One year ago, Pope Francis began his visits ar...   
     AND              It is my honour to represent my country, the P...   
     ARE              It is my pleasure to congratulate Mr. Mogens L...   

                      WordCount  ...  example_sentence  sentence_hedge_ratio  \
Year ISO-alpha3 Code             ...                                           
2015 AFG                   1741  ...              None                   NaN   
     AGO                   1644  ...              None                   NaN   
     ALB                   1097  ...              None                   NaN   
     AND                   2522  ...              None                   NaN   
     ARE                   1673  ...              None                   NaN   

                      sentence_commit_ratio  sentence_net_caution  \
Year ISO-alpha3 Code                                                
2015 AFG                                NaN                   NaN   
     AGO                                NaN                   NaN   
     ALB                                NaN                   NaN   
     AND                                NaN                   NaN   
     ARE        

## Checks: do the methods agree?

The correlation between the three net caution scores, and the most hedged and most committed example sentences.

In [12]:
mask = df['transition_mention_count'] >= 1
net_caution_cols = ['sentence_net_caution', 'word15_net_caution', 'pos_net_caution']

print(df.loc[mask, net_caution_cols].corr())

                      sentence_net_caution  word15_net_caution  \
sentence_net_caution              1.000000            0.677636   
word15_net_caution                0.677636            1.000000   
pos_net_caution                   0.247952            0.132391   

                      pos_net_caution  
sentence_net_caution         0.247952  
word15_net_caution           0.132391  
pos_net_caution              1.000000  


In [13]:
qa_cols = ['Country or Area', 'pos_net_caution', 'example_sentence']

print("Most hedged (top 5 by pos_net_caution):")
print(df.loc[mask].sort_values('pos_net_caution', ascending=False)[qa_cols].head(5))

print()
print("Most committal (bottom 5 by pos_net_caution):")
print(df.loc[mask].sort_values('pos_net_caution', ascending=True)[qa_cols].head(5))

Most hedged (top 5 by pos_net_caution):
                            Country or Area  pos_net_caution  \
Year ISO-alpha3 Code                                           
2021 SMR                         San Marino              1.0   
     ETH                           Ethiopia              1.0   
2022 SEN                            Senegal              1.0   
2015 ISL                            Iceland              1.0   
2025 KNA              Saint Kitts and Nevis              1.0   

                                                       example_sentence  
Year ISO-alpha3 Code                                                     
2021 SMR              Emerging from COVID-19 pandemic could represen...  
     ETH              Hopefully, we have inspired others to develop ...  
2022 SEN              At the same time, we hope to reach a consensus...  
2015 ISL              We need to aim for the elimination of carbon-b...  
2025 KNA              It is guided by seven fundamental pillars, en

## Save

In [14]:
output_columns = [
    'Session', 'Country or Area', 'Region Name', 'Sub-region Name',
    'transition_mention_count',
    'sentence_hedge_count', 'sentence_commit_count',
    'sentence_hedge_ratio', 'sentence_commit_ratio', 'sentence_net_caution',
    'word15_hedge_count', 'word15_commit_count',
    'word15_hedge_ratio', 'word15_commit_ratio', 'word15_net_caution',
    'pos_hedge_count', 'pos_commit_count',
    'pos_hedge_ratio', 'pos_commit_ratio', 'pos_net_caution',
    'example_sentence',
]

df_out = df[output_columns]
df_out.to_parquet('../datasets/speeches_advanced_clean.parquet')

print(df_out.shape)

(2128, 21)


## Final columns

- Year (index)
- ISO-alpha3 Code (index)
- Session
- Country or Area
- Region Name
- Sub-region Name
- transition_mention_count
- sentence_hedge_count
- sentence_commit_count
- sentence_hedge_ratio
- sentence_commit_ratio
- sentence_net_caution
- word15_hedge_count
- word15_commit_count
- word15_hedge_ratio
- word15_commit_ratio
- word15_net_caution
- pos_hedge_count
- pos_commit_count
- pos_hedge_ratio
- pos_commit_ratio
- pos_net_caution
- example_sentence